In [96]:
import ast
import copy
import math
import networkx as nx

from itertools import permutations

In [2]:
with open('data/level_2.txt', 'r') as f:
    file_content = f.read()

In [3]:
file_content

'{\n  "start": "A",\n  "end": "B",\n  "required_stops": ["S1", "S2", "S3", "S4"],\n  "adjacency_list": {\n    "A":   [{"node": "P1",  "time": 4, "risk": 0}, {"node": "P6",  "time": 5, "risk": 2}],\n    "B":   [{"node": "P5",  "time": 4, "risk": 0}, {"node": "P12", "time": 4, "risk": 0}],\n    "S1":  [{"node": "P3",  "time": 4, "risk": 0}, {"node": "P4",  "time": 4, "risk": 1}, {"node": "P9",  "time": 5, "risk": 1}],\n    "S2":  [{"node": "P7",  "time": 4, "risk": 0}, {"node": "P8",  "time": 5, "risk": 1}, {"node": "P10", "time": 5, "risk": 2}],\n    "S3":  [{"node": "P1",  "time": 4, "risk": 0}, {"node": "P2",  "time": 4, "risk": 1}],\n    "S4":  [{"node": "P4",  "time": 5, "risk": 0}, {"node": "P5",  "time": 4, "risk": 0}, {"node": "P10", "time": 7, "risk": 0}],\n    "P1":  [{"node": "A",   "time": 4, "risk": 0}, {"node": "S3",  "time": 4, "risk": 0}],\n    "P2":  [{"node": "S3",  "time": 4, "risk": 1}, {"node": "P3",  "time": 3, "risk": 0}],\n    "P3":  [{"node": "P2",  "time": 3, "r

In [49]:
data = ast.literal_eval(file_content)

In [50]:
data

{'start': 'A',
 'end': 'B',
 'required_stops': ['S1', 'S2', 'S3', 'S4'],
 'adjacency_list': {'A': [{'node': 'P1', 'time': 4, 'risk': 0},
   {'node': 'P6', 'time': 5, 'risk': 2}],
  'B': [{'node': 'P5', 'time': 4, 'risk': 0},
   {'node': 'P12', 'time': 4, 'risk': 0}],
  'S1': [{'node': 'P3', 'time': 4, 'risk': 0},
   {'node': 'P4', 'time': 4, 'risk': 1},
   {'node': 'P9', 'time': 5, 'risk': 1}],
  'S2': [{'node': 'P7', 'time': 4, 'risk': 0},
   {'node': 'P8', 'time': 5, 'risk': 1},
   {'node': 'P10', 'time': 5, 'risk': 2}],
  'S3': [{'node': 'P1', 'time': 4, 'risk': 0},
   {'node': 'P2', 'time': 4, 'risk': 1}],
  'S4': [{'node': 'P4', 'time': 5, 'risk': 0},
   {'node': 'P5', 'time': 4, 'risk': 0},
   {'node': 'P10', 'time': 7, 'risk': 0}],
  'P1': [{'node': 'A', 'time': 4, 'risk': 0},
   {'node': 'S3', 'time': 4, 'risk': 0}],
  'P2': [{'node': 'S3', 'time': 4, 'risk': 1},
   {'node': 'P3', 'time': 3, 'risk': 0}],
  'P3': [{'node': 'P2', 'time': 3, 'risk': 0},
   {'node': 'S1', 'time': 4

In [51]:
start_node = data['start']
end_node = data['end']
required_stops = data['required_stops']
graph = data['adjacency_list']

In [52]:
required_stops

['S1', 'S2', 'S3', 'S4']

In [62]:
graph

{'A': [{'node': 'P1', 'time': 4, 'risk': 0},
  {'node': 'P6', 'time': 5, 'risk': 2}],
 'B': [{'node': 'P5', 'time': 4, 'risk': 0},
  {'node': 'P12', 'time': 4, 'risk': 0}],
 'S1': [{'node': 'P3', 'time': 4, 'risk': 0},
  {'node': 'P4', 'time': 4, 'risk': 1},
  {'node': 'P9', 'time': 5, 'risk': 1}],
 'S2': [{'node': 'P7', 'time': 4, 'risk': 0},
  {'node': 'P8', 'time': 5, 'risk': 1},
  {'node': 'P10', 'time': 5, 'risk': 2}],
 'S3': [{'node': 'P1', 'time': 4, 'risk': 0},
  {'node': 'P2', 'time': 4, 'risk': 1}],
 'S4': [{'node': 'P4', 'time': 5, 'risk': 0},
  {'node': 'P5', 'time': 4, 'risk': 0},
  {'node': 'P10', 'time': 7, 'risk': 0}],
 'P1': [{'node': 'A', 'time': 4, 'risk': 0},
  {'node': 'S3', 'time': 4, 'risk': 0}],
 'P2': [{'node': 'S3', 'time': 4, 'risk': 1},
  {'node': 'P3', 'time': 3, 'risk': 0}],
 'P3': [{'node': 'P2', 'time': 3, 'risk': 0},
  {'node': 'S1', 'time': 4, 'risk': 0}],
 'P4': [{'node': 'S1', 'time': 4, 'risk': 1},
  {'node': 'S4', 'time': 5, 'risk': 0}],
 'P5': [{'

In [63]:
updated_graph = copy.deepcopy(graph)

In [64]:
edges = []
for key, values in updated_graph.items():
    for value in values:
        node = value['node']
        effective_weight = value['time'] + value['risk']
        entry = (key, node, effective_weight)
        edges.append(entry)

In [65]:
edges

[('A', 'P1', 4),
 ('A', 'P6', 7),
 ('B', 'P5', 4),
 ('B', 'P12', 4),
 ('S1', 'P3', 4),
 ('S1', 'P4', 5),
 ('S1', 'P9', 6),
 ('S2', 'P7', 4),
 ('S2', 'P8', 6),
 ('S2', 'P10', 7),
 ('S3', 'P1', 4),
 ('S3', 'P2', 5),
 ('S4', 'P4', 5),
 ('S4', 'P5', 4),
 ('S4', 'P10', 7),
 ('P1', 'A', 4),
 ('P1', 'S3', 4),
 ('P2', 'S3', 5),
 ('P2', 'P3', 3),
 ('P3', 'P2', 3),
 ('P3', 'S1', 4),
 ('P4', 'S1', 5),
 ('P4', 'S4', 5),
 ('P5', 'S4', 4),
 ('P5', 'B', 4),
 ('P6', 'A', 7),
 ('P6', 'P7', 4),
 ('P7', 'P6', 4),
 ('P7', 'S2', 4),
 ('P7', 'P11', 6),
 ('P8', 'S2', 6),
 ('P8', 'P9', 6),
 ('P9', 'P8', 6),
 ('P9', 'S1', 6),
 ('P10', 'S2', 7),
 ('P10', 'S4', 7),
 ('P11', 'P7', 6),
 ('P11', 'P12', 6),
 ('P12', 'P11', 6),
 ('P12', 'B', 4)]

In [66]:
# Create a weighted graph
G = nx.Graph()
G.add_weighted_edges_from(edges)

In [97]:
combinations = {}
max_distance = math.inf
all_perms = permutations(required_stops)
for perm in all_perms:
    route = []
    distance = 0
    travel_path = [start_node] + list(perm) + [end_node]
    for i, (start_loc, end_loc) in enumerate(zip(travel_path[:-1], travel_path[1:])):
        calc_route = nx.dijkstra_path(G, source=start_loc, target=end_loc)
        distance += nx.dijkstra_path_length(G, source=start_loc, target=end_loc)
        if not i:
            route += calc_route
        else:
            # Exclude first entry since its the same as last entry of previous
            # calculated route
            route += calc_route[1:]

    if distance <= max_distance:
        combinations.update({distance: route})
        max_distance = distance

In [98]:
combinations

{91: ['A',
  'P1',
  'S3',
  'P2',
  'P3',
  'S1',
  'P9',
  'P8',
  'S2',
  'P7',
  'P6',
  'A',
  'P1',
  'S3',
  'P2',
  'P3',
  'S1',
  'P4',
  'S4',
  'P5',
  'B'],
 77: ['A',
  'P1',
  'S3',
  'P2',
  'P3',
  'S1',
  'P3',
  'P2',
  'S3',
  'P1',
  'A',
  'P6',
  'P7',
  'S2',
  'P10',
  'S4',
  'P5',
  'B'],
 75: ['A',
  'P6',
  'P7',
  'S2',
  'P8',
  'P9',
  'S1',
  'P3',
  'P2',
  'S3',
  'P2',
  'P3',
  'S1',
  'P4',
  'S4',
  'P5',
  'B'],
 68: ['A',
  'P6',
  'P7',
  'S2',
  'P7',
  'P6',
  'A',
  'P1',
  'S3',
  'P2',
  'P3',
  'S1',
  'P4',
  'S4',
  'P5',
  'B'],
 60: ['A',
  'P1',
  'S3',
  'P2',
  'P3',
  'S1',
  'P9',
  'P8',
  'S2',
  'P10',
  'S4',
  'P5',
  'B']}

In [99]:
# Extract minimum distance
min(combinations.keys())

60

In [100]:
min_distance = min(combinations.keys())
optimum_route = combinations[min_distance]

In [101]:
optimum_route

['A', 'P1', 'S3', 'P2', 'P3', 'S1', 'P9', 'P8', 'S2', 'P10', 'S4', 'P5', 'B']